# 📘 Week 11: Radar Signal Processing
Welcome to Week 11! In this module, we will explore the core signal processing algorithms behind **Frequency Modulated Continuous Wave (FMCW) Radar** systems—specifically focusing on **Range Estimation** and **Velocity Estimation (Range-Doppler Maps)**.

## 🎯 Learning Objectives:
- Mathematical definition and generation of **Linear Frequency-Modulated (LFM) Chirp** signals.
- Simulating a target echo (time delay and attenuation).
- Implementing a mixer to obtain the **Beat Signal** and using FFT to estimate target range.
- Processing multiple chirp pulses to construct and visualize a 2D **Range-Doppler Map**.

## 1. FMCW Radar & Linear Chirps
FMCW radar systems transmit a continuous signal whose frequency changes linearly over time. This is called a **chirp**.
The transmitted chirp $s(t)$ is modeled as:
$$s(t) = \sin\left(2\pi \left(f_0 t + \frac{k}{2} t^2\right)\right)$$
Where:
- $f_0$ is the start (carrier) frequency.
- $k$ is the chirp rate (slope), defined as $\frac{B}{T}$, where $B$ is the bandwidth and $T$ is the chirp duration.

Let's generate and plot a linear chirp signal and its spectrogram.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal

fs = 2000  # 2 kHz sampling rate
T = 1.0    # 1.0 second sweep time
t = np.linspace(0, T, int(fs * T), endpoint=False)

# Chirp parameters
f0 = 50.0   # Start frequency: 50 Hz
f1 = 250.0  # End frequency: 250 Hz
k = (f1 - f0) / T  # Chirp rate

# Transmitted chirp signal
s_tx = np.sin(2 * np.pi * (f0 * t + 0.5 * k * t**2))

# Plot the waveform and its spectrogram
plt.figure(figsize=(12, 5))
plt.subplot(2, 1, 1)
plt.plot(t[:400], s_tx[:400], color='#1f77b4', linewidth=1.5)
plt.title("Transmitted Linear Chirp (First 200 ms)")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.grid(True)

plt.subplot(2, 1, 2)
frequencies, times, spectrogram = signal.spectrogram(s_tx, fs, nperseg=128, noverlap=110)
plt.pcolormesh(times, frequencies, 10 * np.log10(spectrogram + 1e-10), shading='gouraud', cmap='viridis')
plt.title("Spectrogram of the Linear Chirp (Linear Frequency Sweep)")
plt.ylabel("Frequency (Hz)")
plt.xlabel("Time (seconds)")
plt.colorbar(label='Power (dB)')

plt.tight_layout()
plt.show()

## 2. Target Range Estimation using Beat Signals
When the chirp hits a target, it is reflected back. The received echo $r(t)$ is a delayed and attenuated copy of the transmitted signal:
$$r(t) = \alpha \cdot s(t - \tau)$$
Where $\tau$ is the round-trip delay time: $\tau = \frac{2R}{c}$ ($R$ is target range, $c$ is wave speed).

The radar mixer multiplies $s(t)$ and $r(t)$ to obtain the **Beat Signal** $b(t)$:
$$b(t) = s(t) \times r(t)$$
Applying a low-pass filter to $b(t)$ yields a single sinusoidal tone called the **Beat Frequency ($f_b$)**:
$$f_b = k \cdot \tau = k \cdot \frac{2R}{c}$$
By performing an FFT on the beat signal, we can detect the peak beat frequency $f_b$ and solve for target range:
$$R = \frac{f_b \cdot c}{2k}$$

Let's simulate a target at a distance of **85 meters** (using $c = 340$ m/s as speed of sound/waves for easy visual scaling).

In [ ]:
c = 340.0  # Speed of wave propagation (m/s)
target_range = 85.0  # Target distance (meters)

# Calculate round-trip time delay
tau = (2.0 * target_range) / c
print(f"Target Range: {target_range} m")
print(f"Round-trip delay time (tau): {tau:.3f} seconds")

# Generate received echo signal (padded with zeros before delay tau)
num_delay_samples = int(tau * fs)
s_rx = np.zeros_like(s_tx)
s_rx[num_delay_samples:] = 0.6 * np.sin(2 * np.pi * (f0 * (t[num_delay_samples:] - tau) + 0.5 * k * (t[num_delay_samples:] - tau)**2))

# Mix signals (multiply Tx and Rx)
beat_signal = s_tx * s_rx

# Apply FFT to beat signal to find beat frequency peak
N_fft = len(beat_signal)
fft_val = np.fft.fft(beat_signal)
freqs = np.fft.fftfreq(N_fft, 1/fs)

# Limit search to positive frequencies
pos_indices = freqs >= 0
pos_freqs = freqs[pos_indices]
pos_fft = np.abs(fft_val)[pos_indices]

# Find peak frequency
peak_idx = np.argmax(pos_fft)
fb_est = pos_freqs[peak_idx]
range_est = (fb_est * c) / (2.0 * k)

print(f"Detected Beat Frequency: {fb_est:.2f} Hz")
print(f"Estimated Target Range: {range_est:.2f} m (Target: {target_range} m)")

# Plot beat signal and spectrum
plt.figure(figsize=(12, 5))
plt.subplot(2, 1, 1)
plt.plot(t, beat_signal, color='g', label='Beat Signal')
plt.title("Beat Signal (De-ramped Output)")
plt.xlabel("Time (seconds)")
plt.legend()
plt.grid(True)

plt.subplot(2, 1, 2)
plt.plot(pos_freqs, pos_fft, color='r', label='Beat Spectrum')
plt.axvline(x=fb_est, color='k', linestyle='--', label=f'Peak: {fb_est:.1f} Hz ({range_est:.1f} m)')
plt.title("FFT Spectrum of Beat Signal")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 3. Doppler Shift & Range-Doppler Map
If the target is moving, the received signal is also shifted in frequency due to the **Doppler Effect**:
$$f_d = \frac{2 v f_c}{c}$$
Where $v$ is the target velocity and $f_c$ is the carrier frequency.

To estimate both range and velocity, FMCW radars transmit a sequence of chirps (called **pulses**). We arrange the beat signals of multiple chirps into a 2D matrix:
- **Fast-Time (columns)**: Samples within a single chirp (represents range).
- **Slow-Time (rows)**: Pulses across consecutive chirps (represents velocity).

By performing a **2D FFT**, we isolate the target in both range and Doppler velocity, creating a **Range-Doppler Map**.

Let's simulate a moving target at range = **60 meters** with velocity = **15 m/s** over 64 consecutive chirps.

In [ ]:
num_chirps = 64
num_samples = len(t)  # Samples per chirp

c_prop = 340.0  # Speed of wave
v_target = 15.0  # Target velocity (m/s)
r_start = 60.0  # Target start range (meters)

data_matrix = np.zeros((num_chirps, num_samples))

for chirp_idx in range(num_chirps):
    # Compute target range at this specific chirp index
    # sweep time T represents the time interval between consecutive chirps
    t_slow = chirp_idx * T
    current_range = r_start + v_target * t_slow
    
    # Calculate delay
    tau_chirp = (2.0 * current_range) / c_prop
    num_delay = int(tau_chirp * fs)
    
    # Generate Rx echo and mix it with Tx
    s_rx_chirp = np.zeros_like(s_tx)
    s_rx_chirp[num_delay:] = 0.5 * np.sin(2 * np.pi * (f0 * (t[num_delay:] - tau_chirp) + 0.5 * k * (t[num_delay:] - tau_chirp)**2))
    
    data_matrix[chirp_idx, :] = s_tx * s_rx_chirp

# Apply 2D FFT (range-Doppler processing)
# Apply Hamming window in both dimensions to reduce sidelobes
window_fast = np.hamming(num_samples)
window_slow = np.hamming(num_chirps)[:, None]
windowed_matrix = data_matrix * window_fast * window_slow

# Compute 2D FFT and take shift to center Doppler spectrum
rd_map = np.fft.fftshift(np.fft.fft2(windowed_matrix), axes=0)
rd_map_mag = np.abs(rd_map)

# Create range and velocity grids for plotting
# Range limit is bounded by max beat frequency fs/2
max_fb = fs / 2.0
max_range = (max_fb * c_prop) / (2.0 * k)
range_axis = np.linspace(0, max_range, num_samples // 2)

# Doppler velocity is bounded by max slow-time sampling rate (1/T)
# Doppler frequency fd = 2 * v * fc / c -> max velocity
max_doppler_freq = 1.0 / (2.0 * T)
max_velocity = (max_doppler_freq * c_prop) / (2.0 * f0)
velocity_axis = np.linspace(-max_velocity, max_velocity, num_chirps)

# Keep only positive ranges
rd_map_plot = rd_map_mag[:, :num_samples // 2]

# Plot 2D Range-Doppler Map
plt.figure(figsize=(10, 6))
plt.pcolormesh(range_axis, velocity_axis, rd_map_plot, shading='gouraud', cmap='jet')
plt.title("2D Range-Doppler Map")
plt.xlabel("Range (meters)")
plt.ylabel("Velocity (m/s)")
plt.colorbar(label='Magnitude')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## ✅ Reflection & Exercises
- **Windowing Effects:** In Section 3, comment out the windowing arrays (`window_fast` and `window_slow`) and plot the map again. What happens to the width of the target peak and the surrounding grid noise (sidelobes)?
- **Multiple Targets:** How would you modify the simulator in Section 2 to model two targets at different ranges (e.g. at 50 m and 120 m)? What would the resulting spectrum look like?